# Experiment D: Critic Self-Training (Qwen3.5-2B, 4-bit, Multi-Round)
This notebook runs multi-round critic self-training on GSM8K using Qwen/Qwen3.5-2B.

### Key Optimizations Enabled:
1. **Pre-compiled Causal Conv1D**: Loads `/content/drive/MyDrive/wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl` (eliminates unvectorized PyTorch reference loop).
2. **Round-Wise Progression**: Executes rounds outer and arms inner (`vote`, `critic`, `critic_reason`), logging real-time scorecards.
3. **Hardware Acceleration**: Tuned for L4 (24GB) and A100 (40/80GB) with `--batch_size 128` and `--micro_batch_size 8`.
4. **Native Caching**: Automatically skips completed generation, signal scoring, and adapters.

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive, userdata
import os

drive.mount('/content/drive')
# Optional: HF token from Colab Secrets if accessing gated models
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

In [ ]:
# Clone repository or pull latest if already present
import os
if not os.path.exists('/content/SLM'):
    !git clone https://github.com/PrachitDeshinge-324/SLM.git /content/SLM
    %cd /content/SLM
else:
    %cd /content/SLM
    !git fetch origin main
    !git reset --hard origin/main

!git log --oneline -1

In [ ]:
# Install the pre-compiled Causal Conv1D CUDA kernel from Drive
WHEEL_PATH = '/content/drive/MyDrive/wheels/causal_conv1d-1.7.0-cp313-cp313-linux_x86_64.whl'
!pip install -q {WHEEL_PATH}

# Verify CUDA kernel is active
import torch
import causal_conv1d
print('✅ Causal Conv1D CUDA kernel successfully loaded!')

In [ ]:
# Install repository dependencies
!pip install -q -r requirements.txt

# Remove torchao if present (conflicts with peft on Colab)
!pip uninstall -y -q torchao 2>/dev/null || true

# Inspect active GPU
!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv

## 2. Integrity & Wiring Verification

In [ ]:
!PYTHONPATH=. python tests/test_self_training.py
!PYTHONPATH=. python tests/test_signals.py

## 3. Run Critic Self-Training (High Speed)
Executes multi-round self-training across all three arms:
- `vote`: plurality vote over sampled solutions
- `critic`: critic + vote consensus
- `critic_reason`: consensus answers augmented with agreeing critic reasoning traces

In [ ]:
!bash run_critic_self_training.sh \
  --model Qwen/Qwen3.5-2B \
  --precision 4bit \
  --base q2b_4bit_critic_p500 \
  --rounds 2 \
  --pool_limit 500 \
  --eval_limit 100 \
  --batch_size 128 \
  --micro_batch_size 8 \
  --results_dir /content/drive/MyDrive/results

## 4. Evaluation Report & Cross-Arm Scorecard

In [ ]:
!PYTHONPATH=. python scripts/report_critic_self_training.py \
  --results_dir /content/drive/MyDrive/results \
  --model Qwen/Qwen3.5-2B \
  --precision 4bit \
  --base q2b_4bit_critic_p500 \
  --rounds 2

## 5. Teardown / Free GPU

In [ ]:
from google.colab import runtime
runtime.unassign()